In [ ]:
import os
import time
import pickle
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.layers import Dense, Activation, Flatten, Conv2D, MaxPooling2D

ROOT_DIR = Path(os.getcwd()).parent

TRAINING_MODELS_DIR = ROOT_DIR / "notebooks" / "models"
TRAINING_MODELS_DIR.mkdir(parents=True, exist_ok=True)

LOGS_DIR = ROOT_DIR / "notebooks" / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

NAME=f"batch64-epoch15-vs0.3-{int(time.time())}"
tensorboard=TensorBoard(log_dir='logs/{}'.format(NAME))

pickle_in = open("X.pickle","rb")
X = pickle.load(pickle_in)

pickle_in = open("y.pickle","rb")
y = pickle.load(pickle_in)

X = X/255.0

In [ ]:
model = Sequential()

#Layer #1
model.add(Conv2D(256, (3, 3), input_shape=X.shape[1:])) #A convolutional layerl; using a 3x3 window; with the input shape
model.add(Activation('relu')) #A Rectified Linear Unit (ReLU) based activation layer
model.add(MaxPooling2D(pool_size=(2, 2))) #A pooling layer using a pool size of 2x2

#Layer #2 
model.add(Conv2D(256, (3, 3))) #A convolutional layerl; using a 3x3 window
model.add(Activation('relu')) #A Rectified Linear Unit (ReLU) based activation layer
model.add(MaxPooling2D(pool_size=(2, 2))) #A pooling layer using a pool size of 2x2

#Layer #3
model.add(Flatten()) #Flatten the data because conv layer is 2D where Dense wants a 1D dataset
model.add(Dense(64))
model.add(Activation('relu'))

#Output Layer
model.add(Dense(1))
model.add(Activation('sigmoid'))

#Try out optimizer='nadam' for its effiency
model.compile(loss='binary_crossentropy', optimizer='adam',metrics=['accuracy']) 

#Batch sizes between 20 to 200 tend to be optimal
model.fit(X, y, batch_size=64, epochs=15, validation_split=0.3, callbacks=[tensorboard])

In [ ]:
model.summary()

In [ ]:
val_loss, val_acc = model.evaluate(X,y)
print(val_loss, val_acc)

In [ ]:
plt.imshow(X[0], cmap=plt.cm.binary)
plt.show()
print(X[0])

In [ ]:
#Save the model
model.save(f'models/{NAME}.keras')

In [ ]:
#Load the model
new_model = tf.keras.models.load_model(f'models/{NAME}.keras')